In [ ]:
%pip install optuna fastparquet --quiet

In [4]:
import os
import sys
import glob
import shutil

SRC_ROOT = '/kaggle/input/mouse-dynamics-ic'
DST_ROOT = '/kaggle/working/mouse-dynamics'

os.makedirs(DST_ROOT, exist_ok=True)

shutil.copytree(
    f'{SRC_ROOT}/src',
    f'{DST_ROOT}/src',
    dirs_exist_ok=True
)

for src_db in glob.glob(f'{SRC_ROOT}/best-parameters/**/*.db', recursive=True):
    relative = os.path.relpath(src_db, f'{SRC_ROOT}/best-parameters')
    dst_db   = os.path.join(DST_ROOT, 'best-parameters', relative)
    os.makedirs(os.path.dirname(dst_db), exist_ok=True)
    if not os.path.exists(dst_db):
        shutil.copy2(src_db, dst_db)

print(f"src/     copiado : {os.path.exists(f'{DST_ROOT}/src')}")
print(f"db files copiados: {len(glob.glob(f'{DST_ROOT}/best-parameters/**/*.db', recursive=True))}")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/kaggle/input/mouse-dynamics-ic/src'

In [5]:
os.makedirs('/kaggle/working/datasets/raw', exist_ok=True)

src = f'/kaggle/input/datasets/mmoselli/mouse-dynamics-bogazici-split'
dst = f'/kaggle/working/datasets/split/'
if not os.path.exists(dst):
    os.symlink(src, dst)
    
print(f"bogazici-split: exists={os.path.exists(dst)}, symlink={os.path.islink(dst)}")

OSError: [WinError 1314] A required privilege is not held by the client: '/kaggle/input/datasets/mmoselli/mouse-dynamics-bogazici-split' -> '/kaggle/working/datasets/split/'

In [10]:
os.chdir('/kaggle/working/mouse-dynamics')

if '/kaggle/working/mouse-dynamics' not in sys.path:
    sys.path.insert(0, '/kaggle/working/mouse-dynamics')

from src.classifiers    import EnumClassifiers
from src.dataset_loaders import EnumDatasets
from src.preprocessors  import EnumPreprocessors
from src.splitters      import EnumSplitters
from src.orchestrator   import Orchestrator

print("Imports OK")
print(f"  EnumClassifiers : {[e.value for e in EnumClassifiers]}")
print(f"  EnumDatasets    : {[e.value for e in EnumDatasets]}")
print(f"  EnumPreprocessors: {[e.value for e in EnumPreprocessors]}")
print(f"  EnumSplitters   : {[e.value for e in EnumSplitters]}")

Imports OK
  EnumClassifiers : ['random-forest', 'mlp', 'knn']
  EnumDatasets    : ['balabit', 'minecraft', 'bogazici']
  EnumPreprocessors: ['minecraft', 'khan']
  EnumSplitters   : ['minecraft', 'fifty_fifty', 'half']


In [ ]:
from src.classifiers import load_classifier
from src.utils.experiment_logger import ExperimentLogger
from src.utils.parquet_conversor import prepare_extraction_data_from_parquet


ALL_CLASSIFIERS  = [EnumClassifiers.RANDOM_FOREST, EnumClassifiers.MLP, EnumClassifiers.KNN]
ALL_WINDOW_SIZES = [10]
#ALL_WINDOW_SIZES = [10, 50, 100, 150, 200]

total = len(ALL_CLASSIFIERS) * len(ALL_WINDOW_SIZES)
count = 0

for window_size in ALL_WINDOW_SIZES:
    count += len(ALL_CLASSIFIERS)
    print("=" * 80)
    print(f"[{count}/{total}] window_size={window_size}")
    print("=" * 80)

    for classifier in ALL_CLASSIFIERS:
        with ExperimentLogger(
                classifier_name=classifier.value,
                dataset_name=EnumDatasets.BOGAZICI.value,
                preprocessor_name=EnumPreprocessors.KHAN.value,
                splitter_name=EnumSplitters.HALF.value,
                preprocessor_window_size=window_size,
                is_debug=False
        ) as experiment_logger:
            extraction_data = prepare_extraction_data_from_parquet(window_size)
            loaded_classifier = load_classifier(classifier, False)
            loaded_classifier.set_experiment_logger(experiment_logger)
            loaded_classifier.fit(extraction_data)

    print(f"Concluído: window_size={window_size}\n")

print("=" * 80)
print("TODOS OS EXPERIMENTOS CONCLUÍDOS")
print("=" * 80)

[3/3] window_size=10

[ExperimentLogger] run=b2c03702 | random-forest × bogazici | window_size=10 | users=0 skipped=0 | mean_balanced_score=0.0000 | mean_f1_macro=0.0000


FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\datasets\\split'